# 02 - Preserve patient assignments and preprocess

Reuse the original patient assignments for both representations. Fit the existing median/mean/std preprocessing on valid TRAIN timesteps only, apply it unchanged to validation/test, and restore padding zeros. Prepared inputs already contain the fixed V63 encoding; do not encode them again.

**Input contract:** 49 ordered V63 predictors, 23,151 snapshots from 12,447 patients and 1,345 positives; PATIENT_ID + END_DT and RESP remain unchanged. No feature selection, new sampling or population update runs here.

**Availability gate:** reuse existing V63 historical values first. Independent, evidenced timestep availability determines padding; individual feature NaNs use the existing TRAIN-only median imputation. Unknown history stays "Not verified here" and defers execution until Genie verifies the source. No snapshot replay or duplicate encoding is permitted.

**Evaluation limits:** frozen V63 caps/types were originally fitted using RESP, and their upstream population has not been checked against these held-out assignments. TRAIN-only scaling does not resolve that limitation. TEST was previously inspected. Monthly/quarterly and masking are hypotheses; this comparison does not isolate a causal masking effect or establish improved performance.

Use these four notebooks in order. The older build scripts and folder guide describe the previous experiment and must not regenerate this revision. Save sensitive artifacts only through the existing private warehouse path and clear executed outputs before sharing notebook code.


In [ ]:
# Connection and version identifiers
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply the existing private sf_options connection on the approved Spark runtime.')
sf_options_dl_poc = dict(sf_options)
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'H003'
RUN_ID = 'M001'
import re
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use short uppercase identifiers; use matching IDs in all four notebooks.')
# New output names prevent collision with prior saved models; original inputs remain unchanged.
EXPERIMENT_PREFIX = PREFIX + '_BUSINESS49_TEMPORAL_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
REFERENCE_MODEL_TABLE = PREFIX + '_MODEL_RUN_001'
REFERENCE_NAMES = {'checkpoint.pt', 'training_summary.json', 'training_history.csv', 'training_history.png'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device='auto')
print('Fixed V63 business features; source cohort and RESP retained. Run MONTHLY then QUARTERLY.')



IMPLEMENTATION_SHA256 = '5dc8fe6713c610b37af201137d51e2cc6e1d911727e4775d983edc6efae8049d'


In [ ]:
# Embedded input, split and preprocessing checks
"""Embedded notebook helpers: fixed business features, calendar grids and padding."""
import io
import json
import hashlib
import marshal
import numpy as np
import pandas as pd


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ordered_features():
    features, comparison = configured_features()
    require(len(features) == 49, 'V63 MODEL_TYPE.FEATURES must contain exactly 49 predictors.')
    return features, comparison


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()




















PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}


def load_prepared():
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require(manifest['schema'] == 4 and manifest.get('padding_contract') == 'independent_timestep_availability_v1' and manifest['dataset_id'] == DATASET_ID, 'Prepared dataset ID/version changed.')
    require(manifest['implementation_sha256'] == IMPLEMENTATION_SHA256, 'Prepared dataset was produced by different code.')
    features, _ = ordered_features()
    require(features == manifest['features'], 'Authoritative feature order changed after preparation.')
    _, _, current_encoding = load_business_parameters(features)
    require(current_encoding == manifest['business_encoding'], 'Frozen V63 cap/type parameters changed after preparation.')
    metadata = pd.DataFrame(json.loads(blobs['population.json']), columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(json.loads(blobs['population.json']) == snapshot_records(metadata), 'Prepared snapshot order changed.')
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(snapshot_records(metadata)) == manifest['population_sha256'], 'Population hash mismatch.')
    audit = pd.DataFrame(manifest['audit'])
    require(audit.FEATURE_NAME.tolist() == features, 'Misordered feature audit.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.shape == (len(metadata), count, 49) and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions.')
        require(not np.isinf(X).any() and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0),
                'Invalid padding or infinite input; individual NaNs are allowed only in valid timesteps.')
        require(bool(manifest['representations'][name].get('availability_sha256'))
                and bool(manifest['representations'][name].get('provenance')), 'Missing availability/source evidence.')
        require(array_hash(X) == manifest['representations'][name]['X_sha256'] and array_hash(valid) == manifest['representations'][name]['valid_sha256'], 'Tensor/mask fingerprint changed.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    return metadata, features, bundles, manifest


def fit_temporal_preprocessor(X, valid, rows):
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all(), 'Invalid input or nonfinite padding.')
    observed = X[rows][valid[rows]]
    require(len(observed) > 0, 'No observed TRAIN timesteps; cannot fit preprocessing.')
    require(not np.isnan(observed).all(axis=0).any(),
            'Feature has no observed valid TRAIN values; fitted imputation unavailable. Defer this representation.')
    return fit_preprocessor(observed)


def transform_temporal(X, valid, state):
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all(), 'Invalid input or nonfinite padding.')
    # Do not standardize placeholders; transform only genuinely available timesteps.
    values = np.zeros(X.shape, dtype=np.float32)
    if valid.any():
        values[valid], _ = transform_features(X[valid], state)
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
    reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
    metadata, hashes = bind_split(metadata, frozen, reference)
    split_statistics(metadata)
    require(hashes == audit['reference_hashes'] and digest_json(manifest) == audit['manifest_sha256'], 'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']), 'Preprocessing does not reproduce TRAIN-only fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X), 'mask_sha256': array_hash(b['valid'])})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit

def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value


def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out


def bind_split(metadata, frozen, reference):
    original = normalize_metadata(metadata).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized["SPLIT"] = frozen.SPLIT.to_numpy()
    normalized["SPLIT_CONFIG"] = frozen.SPLIT_CONFIG.to_numpy()
    normalized = normalized.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    if not original.equals(normalized[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Saved split differs from the prepared snapshots/labels.")
    if normalized[["SPLIT", "SPLIT_CONFIG"]].isna().any().any():
        raise ValueError("Incomplete frozen split.")
    if set(normalized.SPLIT) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    if normalized.groupby("PATIENT_ID").SPLIT.nunique().gt(1).any():
        raise ValueError("Patient leakage between splits.")
    if normalized.SPLIT_CONFIG.nunique() != 1:
        raise ValueError("Inconsistent split configuration.")
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {"snapshot_manifest_sha256": digest_json(records),
              "split_config_sha256": digest_json(json.loads(normalized.SPLIT_CONFIG.iloc[0]))}
    if reference.get("run_id") != "RUN_001" or reference.get("training_complete") is not True:
        raise ValueError("Expected completed original RUN_001 reference.")
    if any(reference.get("input_hashes", {}).get(k) != v for k, v in hashes.items()):
        raise ValueError("Patient assignments differ from original RUN_001 fingerprints.")
    for _, part in normalized.groupby("SPLIT"):
        if set(part.RESP) != {0, 1}:
            raise ValueError("Each split needs both outcome classes.")
    return normalized, hashes

def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}

def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)
def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())



def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")
import base64
def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]

"""Pure fixed-parameter transformations and date boundaries; no warehouse IO.

These helpers do not reconstruct a feature, choose predictors, infer source
coverage, fit percentiles, infer feature types, or select a modeling population.
"""

import math
from collections.abc import Sequence



def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(len(result) == 49, 'Exactly 49 authoritative predictors are required.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == 49,
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (49 ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    49-feature input list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)
















"""User-supplied V63 lineage and frozen business encodings; no feature selection."""
SOURCE_REGISTRY = [
    ('MEDICAL', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST', 'SERVICE_DATE', 'DX/PX claims; current extract, no historical availability timestamp supplied'),
    ('PHARMACY', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST', 'FILL_DATE', 'RX claims; transaction and counting rules still require column-level mapping'),
    ('PROVIDERS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PROVIDERS_LATEST', '', 'Current specialty/type; not a historical as-of dimension'),
    ('PLANS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PLANS_LATEST', '', 'Current insurance group/segment; not a historical as-of dimension'),
    ('DEMOGRAPHICS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PATIENT_DEMOGRAPHICS_LATEST', '', 'Year of birth for year-based age; patient/YOB columns must be verified'),
    ('CODE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID', '', 'Code descriptions, hierarchy and chronic flags'),
    ('ANNUAL_CODE_COUNTS', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID_ANNUAL_CNT', '', 'Upstream inclusion lineage only; do not rerun selection'),
    ('PROCEDURE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.ALL_PROCEDURES_SIMPLE', '', 'ICD-10-PCS decomposition; not a substitute for visit-sequence features'),
    ('HCP_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.GATREX_TRIGGER.HCP_LOOKUP', '', 'Active NPI/specialty; historical version not supplied'),
    ('DRUG_BASKET', 'TAK861.NARCOLEPSY_MARKET_BASKET_CODES', '', 'Database not supplied; code columns and inclusion rules still required'),
    ('UPSTREAM_UNIVERSE', 'TAK861.TAK861_TX_READY_PATIENT_UNIVERSE_V8_CF', '', 'Lineage only; never substitute for frozen V63 snapshots'),
    ('UPSTREAM_BACKTEST', 'TAK861.TAK861_TX_READY_BACKTEST_UNIVERSE_V8_CF', '', 'Lineage only; never replace the existing TEST assignment'),
]




def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'max_f_50': 'Ceiling only; RND explanation remains a hypothesis until ranked rows establish it'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    required = set(['PATIENT_ID', 'END_DT', 'RESP'] + features)
    require(required.issubset(read_table(SOURCE_PREFIX + '_MODEL_DATA').columns), 'MODEL_DATA lacks configured columns.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'raw reconstructed features', 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract


In [ ]:
# Load verified monthly and quarterly inputs without repeating V63 encoding.
metadata, features, bundles, manifest = load_prepared()
print('Confirmed feature count:', len(features), '; existing V63 encoding retained.')


In [ ]:
# Bind both representations to the original patient split
frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
metadata, reference_hashes = bind_split(metadata, frozen, reference)
statistics = split_statistics(metadata)
display(statistics)
display(pd.DataFrame([{'TRAIN_VALIDATION_OVERLAP': 0, 'TRAIN_TEST_OVERLAP': 0, 'VALIDATION_TEST_OVERLAP': 0,
                       'MONTHLY_QUARTERLY_ASSIGNMENTS_IDENTICAL': True}]))
indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
print('Same source keys, labels and original RUN_001 patient assignments verified; no new splits created.')


In [ ]:
# Fit the existing median/mean/std only on valid TRAIN timesteps.
preprocessing, preprocessing_coverage = {}, []
for name, b in bundles.items():
    state = fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train'])
    preprocessing[name] = state
    values = transform_temporal(b['raw_X'], b['valid'], state)
    require(values.shape == b['raw_X'].shape and np.all(values[~b['valid']] == 0), 'Transformed padding changed.')
    preprocessing_coverage.append({'MODEL': name, 'VALID_TRAIN_TIMESTEPS': int(b['valid'][indices['train']].sum()),
        'IMPUTED_FEATURE_VALUES': int(np.isnan(b['raw_X'][b['valid']]).sum()),
        'ALL_PADDED_SNAPSHOTS': int((~b['valid'].any(axis=1)).sum()), 'ALL_MODEL_VALUES_FINITE': bool(np.isfinite(values).all())})
display(pd.DataFrame(preprocessing_coverage))


In [ ]:
# Save frozen preprocessing and the identical split manifest
split_rows = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
split_audit = {'manifest_sha256': digest_json(manifest), 'reference_hashes': reference_hashes,
               'preprocessing_sha256': digest_json(preprocessing), 'split_summary': statistics.to_dict('records'),
               'fit_scope': 'Valid TRAIN timesteps only, separately for monthly and quarterly',
               'new_assignments_created': False, 'monthly_quarterly_assignments_identical': True}
save_artifacts(SPLIT_TABLE, {'split.json': canonical_json(split_rows).encode(),
    'preprocessing.json': canonical_json(preprocessing).encode(), 'audit.json': canonical_json(split_audit).encode()})
experiments, _, _ = load_experiment()
print('Read-back verified. Continue with notebook 03.')
